# Подготовка данных

## Считаем данные в исходном csv формате

In [4]:
!pip install findspark

Defaulting to user installation because normal site-packages is not writeable


Перезапустить kernel после установки!

In [5]:
import findspark
findspark.init()

In [10]:
!hdfs dfs -mkdir -p /user/ubuntu/data

In [12]:
!hadoop distcp s3a://hw-2-bucket/ user/ubuntu/data

2026-07-20 10:59:15,155 INFO tools.DistCp: Input Options: DistCpOptions{atomicCommit=false, syncFolder=false, deleteMissing=false, ignoreFailures=false, overwrite=false, append=false, useDiff=false, useRdiff=false, fromSnapshot=null, toSnapshot=null, skipCRC=false, blocking=true, numListstatusThreads=0, maxMaps=20, mapBandwidth=0.0, copyStrategy='uniformsize', preserveStatus=[], atomicWorkPath=null, logPath=null, sourceFileListing=null, sourcePaths=[s3a://hw-2-bucket/], targetPath=user/ubuntu/data, filtersFile='null', blocksPerChunk=0, copyBufferSize=8192, verboseLog=false, directWrite=false}, sourcePaths=[s3a://hw-2-bucket/], targetPathExists=true, preserveRawXattrsfalse
2026-07-20 10:59:15,245 INFO impl.MetricsConfig: Loaded properties from hadoop-metrics2.properties
2026-07-20 10:59:15,401 INFO impl.MetricsSystemImpl: Scheduled Metric snapshot period at 10 second(s).
2026-07-20 10:59:15,401 INFO impl.MetricsSystemImpl: JobTracker metrics system started
2026-07-20 10:59:15,491 INFO i

In [6]:
!hdfs dfs -ls user/ubuntu/data

Found 5 items
-rw-r--r--   1 ubuntu hadoop 2807409271 2026-07-20 10:40 user/ubuntu/data/2019-08-22.txt
-rw-r--r--   1 ubuntu hadoop 2854479008 2026-07-20 10:40 user/ubuntu/data/2019-09-21.txt
-rw-r--r--   1 ubuntu hadoop 2895460543 2026-07-20 10:41 user/ubuntu/data/2019-10-21.txt
-rw-r--r--   1 ubuntu hadoop 2939120942 2026-07-20 10:39 user/ubuntu/data/2019-11-20.txt
-rw-r--r--   1 ubuntu hadoop 2995462277 2026-07-20 10:41 user/ubuntu/data/2019-12-20.txt


In [19]:
!hdfs dfs -cat user/ubuntu/data/2019-12-20.txt | head -n 20

# tranaction_id | tx_datetime | customer_id | terminal_id | tx_amount | tx_time_seconds | tx_time_days | tx_fraud | tx_fraud_scenario
187969299,2019-12-20 17:09:06,0,359,92.58,10429746,120,0,0
187969300,2019-12-20 19:52:17,0,359,36.68,10439537,120,0,0
187969301,2019-12-20 14:26:12,1,981,150.38,10419972,120,0,0
187969302,2019-12-20 19:04:11,3,732,12.50,10436651,120,0,0
187969303,2019-12-20 12:35:14,5,502,31.76,10413314,120,0,0
187969304,2019-12-20 17:40:04,6,29,90.15,10431604,120,0,0
187969305,2019-12-20 06:46:42,6,8,38.00,10392402,120,0,0
187969306,2019-12-20 19:02:43,8,312,19.26,10436563,120,0,0
187969307,2019-12-20 00:08:28,8,26,67.58,10368508,120,0,0
187969308,2019-12-20 13:26:41,10,549,22.84,10416401,120,0,0
187969309,2019-12-20 14:24:29,10,380,130.45,10419869,120,0,0
187969310,2019-12-20 15:59:01,11,132,38.31,10425541,120,0,0
187969311,2019-12-20 16:37:42,11,733,47.78,10427862,120,0,0
187969312,2019-12-20 20:05:19,11,294,75.86,10440319,120,0,0
187969313,2019-12-20 10:56:59,11,337,

In [5]:
!hdfs dfs -ls

Found 2 items
drwxr-xr-x   - ubuntu hadoop          0 2026-07-20 10:53 dataset
drwxr-xr-x   - ubuntu hadoop          0 2026-07-20 10:37 user


In [18]:
!hdfs dfs -ls data

Создадим SparkSession

In [7]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("OTUS")
        .getOrCreate()
)

Загрузим данные

In [8]:
df = spark.read.csv("user/ubuntu/data/", inferSchema=True, header=True)

Поделим данные на 10 разделов и сохраним в формате parquet

In [9]:
%%time
df.count()

CPU times: user 5.79 ms, sys: 0 ns, total: 5.79 ms
Wall time: 24.6 s


234964612

In [10]:
(
    df
        .repartition(10)
        .write
        .mode("overwrite")
        .parquet("data/train.parquet")
)

AnalysisException: Attribute name "# tranaction_id | tx_datetime | customer_id | terminal_id | tx_amount | tx_time_seconds | tx_time_days | tx_fraud | tx_fraud_scenario" contains invalid character(s) among " ,;{}()\n\t=". Please use alias to rename it.;

Проверим размер сохраненного файла

In [13]:
!hdfs dfs -ls -h data/train.parquet

Found 11 items
-rw-r--r--   1 ubuntu hadoop          0 2025-12-03 18:08 data/train.parquet/_SUCCESS
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00000-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00001-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00002-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00003-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00004-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.8 M 2025-12-03 18:08 data/train.parquet/part-00005-09842bc8-6262-4feb-a900-789d9cb1e6df-c000.snappy.parquet
-rw-r--r--   1 ubuntu hadoop      1.

In [14]:
df_from_parquet = spark.read.parquet("data/train.parquet")

In [19]:
%%time
df_from_parquet.count()

CPU times: user 1.91 ms, sys: 0 ns, total: 1.91 ms
Wall time: 303 ms


1000000

In [25]:
df_from_parquet.show(5)

+------+-----------+--------+----------+---------------+-----------------+-----------+------------------+---------------------------+------------------------------+
|row_id|  timestamp| user_id|content_id|content_type_id|task_container_id|user_answer|answered_correctly|prior_question_elapsed_time|prior_question_had_explanation|
+------+-----------+--------+----------+---------------+-----------------+-----------+------------------+---------------------------+------------------------------+
| 62896|   96541616| 1400354|      1087|              0|               94|          3|                 0|                    17000.0|                         false|
|104413|   21162902| 2211492|      5262|              0|               95|          0|                 1|                    11000.0|                          true|
|537225|    4538020|10854346|      7880|              0|               26|          1|                 1|                    30000.0|                          true|
|158202|  